# Model 07: True DuCFF (TCN + Transformer Parallel)


In [ ]:
import numpy as np
import pandas as pd
import time
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, Conv1D, Add, Flatten, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [ ]:
# Load preprocessed data
data = np.load('processed_data.npz')
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

TIME_STEPS = X_train.shape[1]
input_shape = (TIME_STEPS, X_train.shape[2])

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")


In [ ]:
def create_ducff(input_shape):
    inputs = Input(shape=input_shape)
    
    # 1. TCN Channel
    # Causal Convolution + Dilated
    tcn_1 = Conv1D(filters=64, kernel_size=2, padding='causal', dilation_rate=1, activation='relu')(inputs)
    tcn_1 = LayerNormalization(epsilon=1e-6)(tcn_1)
    
    tcn_2 = Conv1D(filters=64, kernel_size=2, padding='causal', dilation_rate=2, activation='relu')(tcn_1)
    tcn_2 = LayerNormalization(epsilon=1e-6)(tcn_2)
    
    tcn_3 = Conv1D(filters=64, kernel_size=2, padding='causal', dilation_rate=4, activation='relu')(tcn_2)
    tcn_3 = LayerNormalization(epsilon=1e-6)(tcn_3)
    
    # Residual Connection (skip connection) to match shapes we use a 1x1 conv if needed, but here input feature is 2, output is 64.
    # Let's just flatten the TCN output for later fusion, or pool it.
    tcn_out = GlobalAveragePooling1D()(tcn_3)
    
    # 2. Transformer Encoder Channel (2 Blocks)
    # Block 1
    attn1 = MultiHeadAttention(num_heads=4, key_dim=32)(inputs, inputs)
    attn1 = Dropout(0.1)(attn1)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attn1)
    
    ffn1 = Dense(64, activation='relu')(out1)
    ffn1 = Dense(input_shape[1])(ffn1) # match dimension for residual
    out2 = LayerNormalization(epsilon=1e-6)(out1 + ffn1)
    
    # Block 2
    attn2 = MultiHeadAttention(num_heads=4, key_dim=32)(out2, out2)
    attn2 = Dropout(0.1)(attn2)
    out3 = LayerNormalization(epsilon=1e-6)(out2 + attn2)
    
    ffn2 = Dense(64, activation='relu')(out3)
    ffn2 = Dense(input_shape[1])(ffn2)
    out4 = LayerNormalization(epsilon=1e-6)(out3 + ffn2)
    
    # Pool transformer output to match TCN output dimension (64)
    # We will pass it to a Dense(64) before pooling so it can be added to TCN output (64)
    trans_out = Dense(64, activation='relu')(out4)
    trans_out = GlobalAveragePooling1D()(trans_out)
    
    # 3. Fusion (Element-wise Addition)
    fusion = Add()([tcn_out, trans_out])
    
    # 4. Output Fully Connected Layers
    x = Dense(64, activation='relu')(fusion)
    x = Dropout(0.2)(x)
    outputs = Dense(2, activation='linear')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="True_DUCFF")
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model = create_ducff(input_shape)
model.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
csv_logger = CSVLogger('training_progress_True_DUCFF.csv', append=True)
EPOCHS = 200
BATCH_SIZE = 64

print(f"\n--- Training: True DUCFF ---")
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, csv_logger], verbose=1)


In [ ]:
# Inference Benchmark
single_sample = X_test[0:1]
_ = model.predict(single_sample, verbose=0)
t_arr = []
for _ in range(30):
    t_s = time.time()
    model.predict(single_sample, verbose=0)
    t_arr.append(time.time() - t_s)
avg_inference_ms = np.mean(t_arr) * 1000

# Predict all test data
pred = model.predict(X_test, verbose=0)
np.save('pred_True_DUCFF.npy', pred)

results = {
    'True_DUCFF': {
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'R_Squared (R²)': r2_score(y_test, pred),
        'Speed (ms)': avg_inference_ms
    }
}
display(pd.DataFrame(results).T)
